In [1]:
# This notebook trains models using a single 80/20 split with wealth-included features
import numpy as np
import pandas as pd

df = pd.read_parquet("MYANMAR/DATA/6.MM_TRAIN_80SET.parquet", engine="pyarrow")

In [2]:
#normalize to numeric first
cols_to_numeric = [
    "hv024RegionDivision",
    "hv025TypeOfResidence",
    "hv219SexOfHead",
    "hv270WealthIndex",  
    "hv106_01EducationOfHead",
    "hv115_01MaritalStatus",
    "v717Occupation",
    "745aHouseOwnership",
]
for col in cols_to_numeric:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [3]:
def restore_ml_dtypes(df):
    df = df.copy()

    df["hv220AgeOfHead"] = df["hv220AgeOfHead"].astype("int8")
    df["hv009FamilySize"] = df["hv009FamilySize"].astype("int8")
    df["hv216HouseSize"] = df["hv216HouseSize"].astype("float64")
    df["hv014NoOfChildren"] = df["hv014NoOfChildren"].astype("int8")
    df["HouseholdClusterElevation"] = df["HouseholdClusterElevation"].astype("float64")

    df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]] = df[[
        "hv024RegionDivision",
        "hv025TypeOfResidence",
        "hv219SexOfHead",
        "hv270WealthIndex",
        "hv106_01EducationOfHead",
        "hv115_01MaritalStatus",
        "v717Occupation",
        "745aHouseOwnership",
        "hv247HasBankAccount",
        "v714CurrentlyWorking",
    ]].astype("category")

    df[[
        "EnergyPoor",
    ]] = df[[
        "EnergyPoor",
    ]].astype("int8")

    return df

In [4]:
df_restore = restore_ml_dtypes(df)

In [5]:
print("\nDtypes:")
print(df_restore.dtypes)

print("\nMissing values:")
print(df_restore.isna().sum().sort_values(ascending=False).head())


Dtypes:
hv270WealthIndex             category
hv024RegionDivision          category
hv025TypeOfResidence         category
hv219SexOfHead               category
hv220AgeOfHead                   int8
hv106_01EducationOfHead      category
hv115_01MaritalStatus        category
hv009FamilySize                  int8
hv247HasBankAccount          category
hv216HouseSize                float64
hv014NoOfChildren                int8
v714CurrentlyWorking         category
v717Occupation               category
745aHouseOwnership           category
HouseholdClusterElevation     float64
EnergyPoor                       int8
dtype: object

Missing values:
hv115_01MaritalStatus    4
hv270WealthIndex         0
hv024RegionDivision      0
hv025TypeOfResidence     0
hv219SexOfHead           0
dtype: int64


In [6]:
TARGET = "EnergyPoor"

drop_cols_mepi_wealth_inclusive = [

    # Country (not needed for per-country models)
    "hv000CountryCode",

    # ID variables
    "hhidCaseIdentification",
    "hv001ClusterNumber",
    "hv002HouseholdNumber",

    # Sampling weights
    "hv005SimplilingWeight",
    "weight",
]

# Build clean training dataframe
X_cols = [c for c in df_restore.columns
          if c not in drop_cols_mepi_wealth_inclusive + [TARGET]]

train_df = df_restore[X_cols + [TARGET]]

print("WEALTH-INCLUSIVE SPECIFICATION")
print("Target:", TARGET)
print("Number of features:", len(X_cols))
print("Features used:", X_cols)

WEALTH-INCLUSIVE SPECIFICATION
Target: EnergyPoor
Number of features: 15
Features used: ['hv270WealthIndex', 'hv024RegionDivision', 'hv025TypeOfResidence', 'hv219SexOfHead', 'hv220AgeOfHead', 'hv106_01EducationOfHead', 'hv115_01MaritalStatus', 'hv009FamilySize', 'hv247HasBankAccount', 'hv216HouseSize', 'hv014NoOfChildren', 'v714CurrentlyWorking', 'v717Occupation', '745aHouseOwnership', 'HouseholdClusterElevation']


In [7]:
print("\nDtypes:")
print(train_df.dtypes)

print("\nMissing values:")
print(train_df.isna().sum().sort_values(ascending=False).head())


Dtypes:
hv270WealthIndex             category
hv024RegionDivision          category
hv025TypeOfResidence         category
hv219SexOfHead               category
hv220AgeOfHead                   int8
hv106_01EducationOfHead      category
hv115_01MaritalStatus        category
hv009FamilySize                  int8
hv247HasBankAccount          category
hv216HouseSize                float64
hv014NoOfChildren                int8
v714CurrentlyWorking         category
v717Occupation               category
745aHouseOwnership           category
HouseholdClusterElevation     float64
EnergyPoor                       int8
dtype: object

Missing values:
hv115_01MaritalStatus    4
hv270WealthIndex         0
hv024RegionDivision      0
hv025TypeOfResidence     0
hv219SexOfHead           0
dtype: int64


In [ ]:
from autogluon.tabular import TabularPredictor

LABEL = "EnergyPoor"

MODEL_PATH = "MYANMAR/WEALTH_INCLUSIVE/TRADITIONAL/"

hyperparameters_tr = {
    "GBM": {},   # one LightGBM
    "CAT": {},   # one CatBoost
    "RF": {},    # one Random Forest
    "XT": {}     # one Extra Trees
}

predictor_tr = TabularPredictor(
    label=LABEL,
    problem_type="binary",
    eval_metric="accuracy",
    path=MODEL_PATH,
    verbosity=2
).fit(
    train_data=train_df,
    presets="medium_quality",
    hyperparameters=hyperparameters_tr,
    fit_weighted_ensemble=False,
    num_stack_levels=0
)

In [ ]:
from autogluon.tabular import TabularPredictor
from tabnet_ag_model import NeuralNetTabNet

LABEL = "EnergyPoor"

DL_MODEL_PATH = "MYANMAR/WEALTH_INCLUSIVE/DL/"

hyperparameters_dl = {
    "NN_TORCH": {},   # PyTorch tabular NN
    "FASTAI": {},     # fastai tabular NN
    NeuralNetTabNet: [{}]  # custom TabNet
}

predictor_dl = TabularPredictor(
    label=LABEL,
    problem_type="binary",
    eval_metric="accuracy",
    path=DL_MODEL_PATH,
    verbosity=2
).fit(
    train_data=train_df,
    presets="medium_quality",
    hyperparameters=hyperparameters_dl,
    fit_weighted_ensemble=False,
    num_stack_levels=0
)

In [ ]:
from autogluon.tabular import TabularPredictor
from FTTransformer_ag_model import NeuralNetFTTransformer

LABEL = "EnergyPoor"

DLFT_MODEL_PATH = "MYANMAR/WEALTH_INCLUSIVE/DLFT/"

hyperparameters_ft = {
     NeuralNetFTTransformer: [{
         "use_cuda_if_available": True
     }]  # custom FT-Transformer
}

predictor_dlft = TabularPredictor(
    label=LABEL,
    problem_type="binary",
    eval_metric="accuracy",
    path=DLFT_MODEL_PATH,
    verbosity=2
).fit(
    train_data=train_df,
    presets="medium_quality",
    hyperparameters=hyperparameters_ft,
    fit_weighted_ensemble=False,
    num_stack_levels=0,
    num_gpus=1,      # <-- required to run on GPU
    num_cpus=8
)